In [4]:
#!pip install -e ..

In [1]:
from stable_baselines3 import PPO, TD3
from skopt import gp_minimize, gbrt_minimize 
from sb3_contrib import TQC, RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from rl4greencrab import evaluate_agent, multiConstAction, simulator
import pandas as pd
import numpy as np
from rl4greencrab import plot_agent
import ray
from skopt import gp_minimize, gbrt_minimize 
from skopt.plots import plot_convergence, plot_objective
from rl4greencrab import TwoActNormalized, twoActEnv
import os, warnings, logging

In [4]:
param_df = pd.read_csv('../data/posterior/params.csv')

In [7]:
def evaluateConstAct(x):
    config = {
        'random_start':True,
        'observation_type': 'count-biomass-time',
        'control_randomness': True,
        'param_df': param_df
    }
    env = twoActEnv(config)
    agent = multiConstAction(env=env, action=np.array(x)) # applies same action every time
    m_reward = evaluate_agent(agent=agent, ray_remote=True).evaluate(n_eval_episodes=200)
    
    return - m_reward

In [8]:
%%time
os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"
warnings.filterwarnings("ignore", category=UserWarning, module="gymnasium")
warnings.filterwarnings("ignore", category=FutureWarning, module="ray")
max_action = 3000
ray.init(
    num_cpus=1,
    include_dashboard=False,
    ignore_reinit_error=True,
    logging_level=logging.ERROR,
)
res = gp_minimize(evaluateConstAct, 2*[(0.0, max_action)], n_calls=150, verbose=False)
res.x
ray.shutdown()

CPU times: user 3min 25s, sys: 53.1 s, total: 4min 18s
Wall time: 18min


In [10]:
# save constant action
pd.DataFrame([res.x], columns=["action_1", "action_2"]).to_csv("../data/constant_action/constant_actions_result.csv", index=False)